Building a sophisticated documentation search engine that shows hybrid retrieval, multivector reranking, and production-quality evaluation.

Your search engine will understand both semantic meaning and exact keywords, then use fine-grained reranking to surface the most relevant documentation sections. When someone searches for “how to configure HNSW parameters,” your system should return the exact section with practical examples, not just a page that mentions “HNSW” somewhere.

This mirrors real-world retrieval challenges where users need precise answers from large documentation sets. You’ll implement the complete pipeline: ingestion with smart chunking, hybrid search with dense and sparse signals, and multivector reranking for precision.

## What You’ll Build
- A single collection that stores dense, sparse, and ColBERT multivectors
- A hybrid search pipeline (dense + sparse with server-side fusion)
- ColBERT reranking for fine-grained matches
- An evaluation step with Recall@10, MRR, and P50/P95 latency
## Setup
### Models
- Dense: BAAI/bge-small-en-v1.5 (384-dim) or BAAI/bge-base-en-v1.5 (768-dim)
- Sparse: BM25/TF-IDF or SPLADE
- Multivector: colbert-ir/colbertv2.0 (128-dim tokens)

## Dataset
- Scope: A full documentation set (e.g., Qdrant docs or a library you use)
- Fields: Chunk by doc structure. Keep: page_title, section_title, page_url, section_url, breadcrumbs, chunk_text, prev_section_text, next_section_text, tags.
- Filters: Consider tags or breadcrumbs for faceting.
- Payload example:
```
payload = {
    "page_title": "Configuration Guide",
    "section_title": "HNSW Parameters",
    "page_url": "/docs/guides/configuration/",
    "section_url": "/docs/guides/configuration/#hnsw-parameters",
    "breadcrumbs": ["Guides", "Configuration", "HNSW Parameters"],
    "chunk_text": "The main section content...",
    "prev_section_text": "Previous section for context...",
    "next_section_text": "Next section for context...",
    "tags": ["configuration", "performance", "hnsw"]
}
```




# Build Steps
## Step 1: Initialize Client

In [23]:
! pip install qdrant_client numpy fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 22.5 MB/s eta 0:00:00


In [2]:
from qdrant_client import QdrantClient, models
import os


from google.colab import userdata
client = QdrantClient(url=userdata.get("CLUSTER_ENDPOINT"), api_key=userdata.get("QDRANT_API"))

## Step 2: Collection Design

In [3]:
collection_name = "docs_search"

client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "colbert": models.VectorParams(
            size=128,
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM
            ),
            hnsw_config=models.HnswConfigDiff(m=0)
        )
    },
    sparse_vectors_config={"sparse": models.SparseVectorParams()}
)

UnexpectedResponse: Unexpected Response: 409 (Conflict)
Raw response content:
b'{"status":{"error":"Wrong input: Collection `docs_search` already exists!"},"time":0.063326946}'

## Step 3: Parse and Chunk Documents
Pick a documentation set and parse it into structured sections. Preserve the hierarchy users expect.

Section-based chunking

- Primary unit: one chunk per section
- Context: store adjacent sections in prev_section_text / next_section_text
- Metadata: keep titles, URLs, and breadcrumbs for attribution and navigation

In [4]:
!pip install langchain-text-splitters

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

doc_url = "/content/drive/MyDrive/document.md"

with open(doc_url, "r") as f:
    doc_content = f.read()

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
docs_result = markdown_splitter.split_text(doc_content)

In [7]:
import re

chunks_para_qdrant = []

for i, doc in enumerate(docs_result):
    current_content = doc.page_content
    meta = doc.metadata

    # 1. Construir Breadcrumbs desde la jerarquía
    # Tomamos los valores de los headers presentes en el metadata
    breadcrumbs = [meta.get(f"Header {j}") for j in range(1, 4) if f"Header {j}" in meta]

    # 2. Determinar títulos
    page_title = meta.get("Header 1", "Documentation")
    # El section_title es el header más profundo que tengamos
    section_title = breadcrumbs[-1] if breadcrumbs else page_title

    # 3. Contexto adyacente (Sliding Window)
    prev_text = docs_result[i-1].page_content if i > 0 else ""
    next_text = docs_result[i+1].page_content if i < len(docs_result)-1 else ""

    payload = {
        "page_title": page_title,
        "section_title": section_title,
        "page_url": doc_url,
        "breadcrumbs": breadcrumbs,
        "chunk_text": current_content, # El campo principal para el embedding
        "prev_section_text": prev_text,
        "next_section_text": next_text,
        "tags": ["documentation", "manual"]
    }

    chunks_para_qdrant.append(payload)

## Step 4: Embed and Ingest
Embed and upload points with all three vectors and the payload fields you need for display and filters.

Sample Models to Test:

- Dense (Primary Retrieval): Use BAAI/bge-small-en-v1.5 for speed or BAAI/bge-base-en-v1.5 for higher quality. For multilingual documentation, consider intfloat/multilingual-e5-base.
- Multivector (Reranking): Implement late-interaction scoring with ColBERTv2. This provides token-level precision for distinguishing between similar sections.
- Sparse (Lexical): Start with BM25-style sparse weights for exact keyword matching. Optionally experiment with the SPLADE encoder.

In [25]:
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL = "prithivida/Splade_PP_en_v1"
MULTIVECTOR_MODEL = "colbert-ir/colbertv2.0"

In [9]:
from google.colab import userdata
from huggingface_hub import login

# Obtiene el token de los secretos y hace login automáticamente
login(userdata.get('HF_TK'))

In [1]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding

dense_model = TextEmbedding(DENSE_MODEL)
sparse_model = SparseTextEmbedding(SPARSE_MODEL)
colbert_model = LateInteractionTextEmbedding(MULTIVECTOR_MODEL)

# Extract only the 'chunk_text' from each chunk for embedding
texts_to_embed = [chunk["chunk_text"] for chunk in chunks_para_qdrant]

dense_embeds = list(dense_model.embed(texts_to_embed, parallel=0))
sparse_embeds = list(sparse_model.embed(texts_to_embed, parallel=0))
colbert_multivectors = list(colbert_model.embed(texts_to_embed, parallel=0))

points = []

for i, chunk in enumerate(chunks_para_qdrant):
    # For individual encoding within the loop, ensure to use the 'chunk_text'
    dense_vector = dense_embeds[i] # Use pre-computed embeds to avoid re-encoding
    sparse_vector = sparse_embeds[i]
    multi_vector = colbert_multivectors[i]

    points.append(models.PointStruct(
        id=i,
        vector={
            "dense": dense_vector,
            "sparse": sparse_vector,
            "multi": multi_vector
        },
        payload=chunk
    ))

client.upload_points(collection_name=collection_name, points=points)

NameError: name 'DENSE_MODEL' is not defined

## Step 5: Search Pipeline
Convert user queries into the three vector representations needed for hybrid search. Use Qdrant’s Universal Query API to combine dense and sparse with server-side fusion.

Multistage pipeline

- Stage 1 – Hybrid retrieval: dense + sparse with RRF (or DBSF). Retrieve 50–200 candidates to get good recall.
- Stage 2 – Multivector reranking: apply token-level late interaction (MAX_SIM) to the candidates.

In [21]:
research_query = "Wich is the best technique for prompt engineering?"


research_query_dense = next(dense_model.query_embed(research_query))
research_query_sparse = next(sparse_model.query_embed(research_query)).as_object()
research_query_colbert = next(colbert_model.query_embed(research_query))

AttributeError: 'SentenceTransformer' object has no attribute 'query_embed'

In [ ]:
# Prefetch queries - global filter will be automatically applied to both
hybrid_query = [
    # Dense retrieval: semantic understanding
    models.Prefetch(query=research_query_dense, using="dense", limit=100),
    # Sparse retrieval: exact technical term matching
    models.Prefetch(query=research_query_sparse, using="sparse", limit=100),
]

# Fusion stage - combines dense and sparse results
fusion_query = models.Prefetch(
    prefetch=hybrid_query,
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=100,
)

In [ ]:
# The Universal Query: Global filter propagates through all stages
response = client.query_points(
    collection_name=collection_name,
    prefetch=fusion_query,
    query=research_query_colbert,
    using="colbert",
    limit=10,
    with_payload=True,
)

## Step 6: Result Formatting
Transform raw results into user-friendly output: page title, section title, URLs, scores, and a short contextual snippet that explains the match.



In [ ]:
# Display results
print("Top Research Papers:")
for i, hit in enumerate(response.points or [], 1):
    paper = hit.payload
    print(f"{i}. {paper['title']}")
    print(f"   Authors: {', '.join(paper['authors'][:3])}{'...' if len(paper['authors']) > 3 else ''}")
    print(f"   Published: {paper['published_date']} | Citations: {paper['citation_count']}")
    print(f"   Research Area: {paper['research_area']}")
    print(f"   Open Access: {paper['open_access']}")
    print(f"   Score: {hit.score:.4f}\n")

## Step 7: Analyze Your Results
Build a small eval set and measure quality and latency. Use results to guide tuning (fusion strategy, candidate sizes, search-time hnsw_ef, etc.).

Ground Truth:

Create 20–30 realistic queries with expected section URLs/anchors.
Aim for diverse query types that cover how-to, concepts, API usage, and troubleshooting.